# Proyek Akhir: Menyelesaikan Permasalahan Institusi Pendidikan

**Nama:** Bilanawati  
**Username Dicoding:** bilanawati_99  
**Email:** bilanamaulia@gmail.com

---

## 1. Business Understanding

### Latar Belakang
Jaya Jaya Institut merupakan institusi pendidikan tinggi yang telah berdiri sejak tahun 2000. Meskipun telah mencetak banyak lulusan dengan reputasi baik, institusi ini menghadapi permasalahan serius: tingginya angka **dropout** mahasiswa yang mencapai **32.12%** dari total mahasiswa.

Tingginya angka dropout berdampak negatif terhadap reputasi institusi, pendapatan, serta akreditasi. Oleh karena itu, dibutuhkan sistem prediksi berbasis machine learning untuk mendeteksi sedini mungkin mahasiswa yang berpotensi dropout agar dapat diberikan intervensi dan bimbingan khusus.

### Permasalahan Bisnis
1. Bagaimana mengidentifikasi faktor-faktor utama yang berkontribusi terhadap dropout mahasiswa?
2. Bagaimana membangun model machine learning yang dapat memprediksi status mahasiswa (Dropout / Enrolled / Graduate) secara akurat?
3. Bagaimana menyediakan dashboard monitoring yang memudahkan pihak institusi memantau performa mahasiswa?

### Cakupan Proyek
- **Exploratory Data Analysis (EDA)**: Memahami distribusi data dan hubungan antar fitur
- **Data Preprocessing**: Penanganan outlier, encoding, scaling
- **Machine Learning Modeling**: Pelatihan dan evaluasi model prediksi
- **Business Dashboard**: Visualisasi data untuk monitoring performa mahasiswa
- **Deployment**: Prototype Streamlit yang dapat diakses secara online

### Persiapan

**Sumber data:** Dataset performa mahasiswa Jaya Jaya Institut (`data.csv`) — 4.424 baris, 37 kolom.

**Setup environment:**
```bash
pip install -r requirements.txt
```

## 2. Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
import joblib

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
print('Libraries loaded successfully!')

## 3. Data Understanding

Dataset terdiri dari **4.424 record** mahasiswa dengan **37 fitur** meliputi:
- **Demografi**: Marital_status, Age_at_enrollment, Gender, Nacionality, International
- **Akademik**: Previous_qualification_grade, Admission_grade, unit mata kuliah semester 1 & 2
- **Ekonomi**: Tuition_fees_up_to_date, Debtor, Scholarship_holder
- **Makroekonomi**: Unemployment_rate, Inflation_rate, GDP
- **Target**: Status (Dropout / Enrolled / Graduate)

In [2]:
df = pd.read_csv('data.csv', sep=';')

print(f'Shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nStatus Distribution:')
print(df['Status'].value_counts())

Shape: (4424, 37)
Missing values: 0

Status Distribution:
Graduate    2209
Dropout     1421
Enrolled     794
Name: Status, dtype: int64


In [3]:
df.head()

In [4]:
df.describe()

## 4. Exploratory Data Analysis (EDA)

### 4.1 Distribusi Status Mahasiswa
Kita mulai dengan melihat distribusi target variable (Status) untuk memahami ketidakseimbangan kelas.

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
status_counts = df['Status'].value_counts()
colors = ['#e74c3c', '#f39c12', '#2ecc71']
bars = axes[0].bar(status_counts.index, status_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribusi Status Mahasiswa', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Jumlah Mahasiswa')
for bar, val in zip(bars, status_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proporsi Status Mahasiswa', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('status_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nInsight: 32.1% mahasiswa (1.421 orang) melakukan dropout - masalah signifikan yang perlu ditangani!')

### 4.2 Analisis Faktor Keuangan

Faktor keuangan sering menjadi penyebab utama dropout. Kita analisis hubungan status pembayaran uang kuliah dan kepemilikan beasiswa terhadap status mahasiswa.

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tuition fees vs Status
tuition_status = pd.crosstab(df['Tuition_fees_up_to_date'], df['Status'], normalize='index') * 100
tuition_status.index = ['Nunggak', 'Lunas']
tuition_status.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white', linewidth=1.2, rot=0)
axes[0].set_title('Status Mahasiswa berdasarkan\nPembayaran Uang Kuliah', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Status Pembayaran')
axes[0].set_ylabel('Persentase (%)')
axes[0].legend(title='Status')

# Scholarship vs Status
scholar_status = pd.crosstab(df['Scholarship_holder'], df['Status'], normalize='index') * 100
scholar_status.index = ['Non-Beasiswa', 'Penerima Beasiswa']
scholar_status.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white', linewidth=1.2, rot=0)
axes[1].set_title('Status Mahasiswa berdasarkan\nKepemilikan Beasiswa', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Kepemilikan Beasiswa')
axes[1].set_ylabel('Persentase (%)')
axes[1].legend(title='Status')

plt.tight_layout()
plt.savefig('financial_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Mahasiswa yang nunggak uang kuliah memiliki dropout rate hingga 86%!')
print('Insight: Penerima beasiswa memiliki graduate rate jauh lebih tinggi.')

### 4.3 Analisis Performa Akademik

Performa akademik merupakan indikator penting dalam memprediksi dropout. Kita analisis perbandingan nilai rata-rata unit kurikulum antar status mahasiswa.

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

academic_cols = [
    ('Curricular_units_2nd_sem_approved', 'Unit Disetujui Sem 2'),
    ('Curricular_units_2nd_sem_grade', 'Nilai Rata-rata Sem 2'),
    ('Curricular_units_1st_sem_approved', 'Unit Disetujui Sem 1'),
    ('Curricular_units_1st_sem_grade', 'Nilai Rata-rata Sem 1'),
]

for idx, (col, title) in enumerate(academic_cols):
    ax = axes[idx//2][idx%2]
    for i, (status, color) in enumerate(zip(['Dropout', 'Enrolled', 'Graduate'], colors)):
        data = df[df['Status'] == status][col]
        ax.hist(data, bins=20, alpha=0.6, color=color, label=status, edgecolor='none')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Nilai')
    ax.set_ylabel('Frekuensi')
    ax.legend()

plt.suptitle('Distribusi Performa Akademik berdasarkan Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('academic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Mahasiswa dropout cenderung memiliki nilai 0 pada semester — tidak mengikuti evaluasi!')

### 4.4 Analisis Demografi

Kita analisis pengaruh usia saat pendaftaran dan gender terhadap status mahasiswa.

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution
for status, color in zip(['Dropout', 'Enrolled', 'Graduate'], colors):
    data = df[df['Status'] == status]['Age_at_enrollment']
    axes[0].hist(data, bins=30, alpha=0.6, color=color, label=status, edgecolor='none')
axes[0].set_title('Distribusi Usia Saat Pendaftaran\nberdasarkan Status', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Usia')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()
axes[0].axvline(x=25, color='black', linestyle='--', alpha=0.5, label='Usia 25')

# Gender vs Status
gender_status = pd.crosstab(df['Gender'], df['Status'], normalize='index') * 100
gender_status.index = ['Perempuan', 'Laki-laki']
gender_status.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white', linewidth=1.2, rot=0)
axes[1].set_title('Status Mahasiswa berdasarkan Gender', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Persentase (%)')
axes[1].legend(title='Status')

plt.tight_layout()
plt.savefig('demographic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

avg_age = df.groupby('Status')['Age_at_enrollment'].mean()
print('Rata-rata usia saat pendaftaran:')
print(avg_age)
print('\nInsight: Mahasiswa yang lebih tua cenderung dropout — kemungkinan karena beban kerja/keluarga.')

### 4.5 Correlation Heatmap

Kita analisis korelasi antar fitur numerik untuk mendeteksi multikolinearitas.

In [9]:
# Top 15 features correlation
top_features = [
    'Curricular_units_2nd_sem_approved', 'Curricular_units_2nd_sem_grade',
    'Curricular_units_1st_sem_approved', 'Curricular_units_1st_sem_grade',
    'Admission_grade', 'Previous_qualification_grade', 'Age_at_enrollment',
    'Tuition_fees_up_to_date', 'Scholarship_holder', 'Debtor',
    'Curricular_units_2nd_sem_evaluations', 'Curricular_units_1st_sem_evaluations',
    'GDP', 'Unemployment_rate', 'Gender'
]

plt.figure(figsize=(12, 10))
corr_matrix = df[top_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap - Fitur Utama', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Nilai semester 1 dan 2 sangat berkorelasi — keduanya penting untuk prediksi.')

## 5. Data Preprocessing

### 5.1 Encoding Target Variable

Target variable `Status` perlu dikonversi ke format numerik menggunakan Label Encoding:
- `Dropout` → 0
- `Enrolled` → 1  
- `Graduate` → 2

In [10]:
le = LabelEncoder()
df['Status_encoded'] = le.fit_transform(df['Status'])

print('Label Encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {cls} → {i}')

X = df.drop(['Status', 'Status_encoded'], axis=1)
y = df['Status_encoded']

print(f'\nFeatures shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nClass distribution:')
print(pd.Series(y).value_counts().sort_index())

### 5.2 Train-Test Split

Dataset dibagi menjadi 80% training dan 20% testing dengan stratifikasi untuk menjaga proporsi kelas.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'\nTraining class distribution:')
print(pd.Series(y_train).value_counts().sort_index())

## 6. Machine Learning Modeling

### 6.1 Model Training: Random Forest Classifier

Kita menggunakan **Random Forest** dengan alasan:
- Robust terhadap outlier dan missing values
- Mampu menangani data dengan skala berbeda tanpa normalisasi penuh
- Memberikan feature importance untuk interpretasi
- Performa yang baik untuk klasifikasi multi-kelas

Model dibungkus dalam **Pipeline** bersama StandardScaler untuk memastikan konsistensi preprocessing.

In [12]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight='balanced',  # Menangani class imbalance
        n_jobs=-1
    ))
])

pipeline.fit(X_train, y_train)
print('Model berhasil dilatih!')

### 6.2 Evaluasi Model

In [13]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'Accuracy: {acc:.2%}\n')
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 76.84%

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.82      0.75      0.78       284
    Enrolled       0.57      0.36      0.44       159
    Graduate       0.78      0.93      0.85       442

    accuracy                           0.77       885
   macro avg       0.72      0.68      0.69       885
weighted avg       0.76      0.77      0.75       885


In [14]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

# Cross Validation
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
axes[1].bar(range(1, 6), cv_scores, color='#3498db', edgecolor='white', linewidth=1.5)
axes[1].axhline(y=cv_scores.mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {cv_scores.mean():.4f}')
axes[1].set_title('5-Fold Cross Validation Scores', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].set_ylim(0.7, 0.85)

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'CV Scores: {cv_scores}')
print(f'CV Mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

### 6.3 Feature Importance

Analisis feature importance membantu kita memahami faktor-faktor paling berpengaruh dalam prediksi status mahasiswa.

In [15]:
feat_imp = pd.Series(
    pipeline.named_steps['model'].feature_importances_,
    index=X.columns
).sort_values(ascending=True).tail(15)

plt.figure(figsize=(10, 7))
colors_imp = ['#e74c3c' if i >= 12 else '#3498db' for i in range(len(feat_imp))]
bars = plt.barh(feat_imp.index, feat_imp.values, color=colors_imp, edgecolor='white', linewidth=0.8)
plt.title('Top 15 Feature Importance\n(Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 Fitur Paling Berpengaruh:')
top5 = feat_imp.sort_values(ascending=False).head(5)
for feat, imp in top5.items():
    print(f'  {feat}: {imp:.4f} ({imp*100:.2f}%)')

### 6.4 Save Model

In [16]:
import os
os.makedirs('model', exist_ok=True)

joblib.dump(pipeline, 'model/model.pkl')
joblib.dump(le, 'model/label_encoder.pkl')

print('Model saved to model/model.pkl')
print('Label encoder saved to model/label_encoder.pkl')

# Verify
loaded_model = joblib.load('model/model.pkl')
loaded_le = joblib.load('model/label_encoder.pkl')
test_pred = loaded_model.predict(X_test[:5])
print(f'\nVerification prediction: {loaded_le.inverse_transform(test_pred)}')

Model saved to model/model.pkl
Label encoder saved to model/label_encoder.pkl


## 7. Conclusion

### Temuan Utama

1. **Angka Dropout Tinggi**: 32.12% mahasiswa (1.421 orang) melakukan dropout — ini merupakan masalah serius.

2. **Faktor Akademik adalah Prediktor Terkuat**: Unit kurikulum yang disetujui dan nilai semester 1 & 2 merupakan prediktor paling signifikan. Mahasiswa yang tidak mengikuti evaluasi (nilai=0) hampir pasti dropout.

3. **Faktor Keuangan Kritis**: Mahasiswa yang nunggak uang kuliah memiliki dropout rate **86%**. Sebaliknya, penerima beasiswa memiliki graduate rate yang jauh lebih tinggi.

4. **Usia Berpengaruh**: Mahasiswa yang lebih tua saat mendaftar cenderung lebih berisiko dropout, kemungkinan karena beban kerja atau keluarga.

5. **Performa Model**: Random Forest mencapai akurasi **76.84%** dengan CV score **77.4% ± 0.79%** — stabil dan cukup andal untuk digunakan sebagai sistem early warning.

### Rekomendasi Action Items

1. **Sistem Early Warning Akademik**: Implementasikan monitoring otomatis mahasiswa yang tidak mengikuti evaluasi di semester pertama sebagai red flag utama. Jika mahasiswa tidak lulus satupun unit di semester 1, segera lakukan intervensi.

2. **Program Bantuan Keuangan Proaktif**: Identifikasi mahasiswa yang menunggak UKT sejak awal semester dan tawarkan program cicilan, beasiswa darurat, atau keringanan biaya sebelum mereka memutuskan dropout.

3. **Mentoring untuk Mahasiswa Berisiko Tinggi**: Berikan program mentoring dan konseling khusus untuk mahasiswa berusia >25 tahun, mahasiswa dengan IPK rendah, dan mahasiswa yang teridentifikasi berisiko tinggi oleh model ML.

4. **Expand Program Beasiswa**: Data menunjukkan penerima beasiswa memiliki tingkat kelulusan jauh lebih tinggi. Perluas program beasiswa prestasi dan beasiswa berbasis kebutuhan ekonomi.

5. **Dashboard Real-time**: Gunakan business dashboard yang telah dibuat untuk monitoring performa mahasiswa secara berkala, minimal setiap awal dan akhir semester.